In [1]:
!nvidia-smi

Sat May  9 14:39:27 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   50C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
%%writefile vector_add.cu
#include <stdio.h>
__global__ void vector(float *A,float *B, float *C, int N)
{
    int i=blockIdx.x * blockDim.x+threadIdx.x;
    if(i<N)
    {
        C[i]=A[i]+B[i];
    }
}
int main()
{
    int N=4;
    size_t size=N*sizeof(float);
    float A[]={1,2,3,4};
    float B[]={5,6,7,8};
    float C[4];

    float *d_A, *d_B, *d_C;

    cudaMalloc(&d_A,size);
    cudaMalloc(&d_B,size);
    cudaMalloc(&d_C,size);

    cudaMemcpy(d_A,A,size,cudaMemcpyHostToDevice);
    cudaMemcpy(d_B,B,size,cudaMemcpyHostToDevice);

    vector<<<1,N>>>(d_A,d_B,d_C,N);

    cudaMemcpy(C,d_C,size,cudaMemcpyDeviceToHost);

    for(int i=0;i<N;i++)
      printf("%f\n",C[i]);

    cudaFree(d_A);
    cudaFree(d_B);
    cudaFree(d_C);

    return 0;

}

Writing vector_add.cu


In [3]:
!nvcc vector_add.cu -o vector_add

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [4]:
!./vector_add

6.000000
8.000000
10.000000
12.000000


In [5]:
%%writefile matrix_mul.cu

#include <stdio.h>

#define N 2

__global__ void multiply(int A[N][N], int B[N][N], int C[N][N])
{
    int row = threadIdx.y;
    int col = threadIdx.x;

    C[row][col] = 0;

    for(int k = 0; k < N; k++)
    {
        C[row][col] += A[row][k] * B[k][col];
    }
}

int main()
{
    int A[N][N] = {{1,2},{3,4}};
    int B[N][N] = {{5,6},{7,8}};
    int C[N][N];

    int (*d_A)[N], (*d_B)[N], (*d_C)[N];

    cudaMalloc(&d_A, sizeof(A));
    cudaMalloc(&d_B, sizeof(B));
    cudaMalloc(&d_C, sizeof(C));

    cudaMemcpy(d_A, A, sizeof(A), cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, B, sizeof(B), cudaMemcpyHostToDevice);

    dim3 threadsPerBlock(N, N);

    multiply<<<1, threadsPerBlock>>>(d_A, d_B, d_C);

    cudaMemcpy(C, d_C, sizeof(C), cudaMemcpyDeviceToHost);

    printf("Result Matrix:\n");

    for(int i = 0; i < N; i++)
    {
        for(int j = 0; j < N; j++)
        {
            printf("%d ", C[i][j]);
        }

        printf("\n");
    }

    cudaFree(d_A);
    cudaFree(d_B);
    cudaFree(d_C);

    return 0;
}

Writing matrix_mul.cu


In [6]:
!nvcc matrix_mul.cu -o matrix_mul

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [7]:
!./matrix_mul

Result Matrix:
19 22 
43 50 
